# f4_m09_conclusiones_eda.ipynb
## Conclusiones del EDA Final — Puente hacia Fase 5


**TFM: Pronóstico del Éxito y del Abandono en los Títulos de Grado de la UJI**

| | |
|---|---|
| **Autora** | María José Morte Ruiz |
| **Institución** | UOC + Universitat Jaume I |
| **Email** | mjmorteruiz@uoc.edu · morte@uji.es |

---

### Qué hace
Consolida todos los hallazgos de Fase 4 (M01–M08) en decisiones accionables para el modelado.  
No transforma datos ni entrena modelos. Solo documenta y justifica decisiones.

### Requisitos
- `data/automl/dataset_final_tfm.parquet` — dataset de producción (D_strict, 33.621 × 20)
- Resultados de M06 (correlaciones, mutual info, ranking AutoML)
- Resultados de M08 (d de Cohen por feature)

### Genera
- `docs/html/fase4/m09_conclusiones_eda.html`

### Flujo
M01 → M02 → M03 → M04 → M05 → M06 → M07 → M08 → **M09** → Fase 5

### Siguiente
`f5_m01_preparacion.ipynb` — pipeline de preprocesamiento + train/test split


In [1]:
# ── CSS + JS para botones Interpretar ────────────────────────────────────────
js_css_html = (
    '<style>'
    '.ibt{display:inline-flex;align-items:center;gap:5px;padding:5px 12px;'
    'background:#3182ce;color:#fff;border:none;border-radius:5px;'
    'font-size:12px;font-weight:600;cursor:pointer;float:right;margin-top:-2px;}'
    '.ibt:hover{background:#2b6cb0;}'
    '.ipn{display:none;background:#f7fafc;border:1px solid #e2e8f0;'
    'border-radius:6px;padding:14px 16px;margin-top:8px;font-size:13px;'
    'color:#2d3748;line-height:1.6;}'
    '.ipn.vis{display:block;}'
    '.tg{background:#c6f6d5;color:#276749;padding:1px 6px;border-radius:3px;font-weight:600;}'
    '.tb{background:#fed7d7;color:#9b2c2c;padding:1px 6px;border-radius:3px;font-weight:600;}'
    '.tm{background:#fefcbf;color:#744210;padding:1px 6px;border-radius:3px;font-weight:600;}'
    '</style>'
    '<script>'
    'function tg(id){'
    'var p=document.getElementById(id);'
    'var b=document.querySelector("[data-pid=\'" + id + "\']");'
    'if(!p||!b)return;'
    'var vis=p.classList.toggle("vis");'
    'b.textContent=vis?"✖ Cerrar":"📖 Interpretar";}'
    '</script>'
)

def h2btn(titulo, pid):
    return (
        f'<div style="display:flex;align-items:center;justify-content:space-between;'
        f'margin:28px 0 8px;">'
        f'<h2 style="font-size:18px;font-weight:700;color:#1a202c;margin:0;">{titulo}</h2>'
        f'<button class="ibt" data-pid="{pid}" onclick="tg(\'{pid}\')">' 
        f'📖 Interpretar</button></div>'
    )

def panel(pid, texto):
    return f'<div id="{pid}" class="ipn">{texto}</div>'


In [2]:
# ============================================================================
# CELDA 1: IMPORTS Y CONFIGURACIÓN
# ============================================================================
import sys, warnings
warnings.filterwarnings('ignore')
from pathlib import Path

import numpy  as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express       as px
from scipy.stats import pointbiserialr
from sklearn.feature_selection import mutual_info_classif

ROOT = Path.cwd()
for _ in range(6):
    if (ROOT / 'src').exists(): break
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.config import RUTA_AUTOML, RUTA_FEATURES, RUTA_HTML, info_entorno
from src.utils  import crear_directorios, formato_numero_es
from src.html   import generar_html_navegacion_completa, guardar_html
from src.html.render import render_base_html

RUTA_FASE4_HTML = RUTA_HTML / 'fase4'
RUTA_RANKING    = RUTA_AUTOML / 'feature_ranking_m06.parquet'
RUTA_DATASET    = RUTA_AUTOML / 'dataset_final_tfm.parquet'
crear_directorios([RUTA_FASE4_HTML])

TARGET     = 'abandono'
COLOR_ABND = '#e53e3e'
COLOR_EXIT = '#3182ce'
COLOR_WARN = '#d69e2e'
COLOR_OK   = '#38a169'
TITULO     = 'Conclusiones EDA Final'
SUBTITULO  = 'Fase 4 · EDA Final · Módulo 9'

info_entorno()
print('Imports OK')


✓ Directorios verificados: 1
✓ ===========================================================================
✓ 📌 INFORMACIÓN DEL ENTORNO DEL PROYECTO
✓ ===========================================================================
✓ 🖥️  Entorno detectado: Local
✓ 📂 Ruta base:     C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI
✓ 📁 RAW:           C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\00_raw
✓ 📁 INTERIM:       C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\01_interim
✓ 📁 PROCESSED:     C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\02_processed
✓ 📁 FEATURES:      C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\03_features
✓ 📁 AUTOML:        C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\automl
✓ 📁 NOTEBOOKS:     C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\notebooks
✓ 📄 Excel principal: C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\00_raw\datos_proyecto_sin_preinscrip.xlsx
✓

In [3]:
# ============================================================================
# CELDA 2: CARGA DEL DATASET Y VERIFICACIÓN
# ============================================================================
df = pd.read_parquet(RUTA_DATASET)

TARGET    = 'abandono'
FEATURES  = [c for c in df.columns if c != TARGET]
FEAT_NUM  = [f for f in FEATURES if df[f].dtype in ['float64','int64','float32','int32']]
FEAT_CAT  = [f for f in FEATURES if f not in FEAT_NUM]

n_total   = len(df)
n_abnd    = df[TARGET].sum()
n_exit    = n_total - n_abnd
pct_abnd  = n_abnd / n_total * 100
ratio_bal = n_exit / n_abnd   # ratio mayoritaria/minoritaria

print(f'Dataset: {df.shape[0]:,} filas × {df.shape[1]} columnas')
print(f'Features: {len(FEATURES)} ({len(FEAT_NUM)} numéricas, {len(FEAT_CAT)} categóricas)')
print(f'Target: {n_abnd:,} abandono ({pct_abnd:.1f}%) · {n_exit:,} éxito ({100-pct_abnd:.1f}%)')
print(f'Ratio desbalanceo: 1:{ratio_bal:.1f}  ({"desbalanceo moderado" if ratio_bal < 4 else "desbalanceo alto"})')
print(f'Nulos: {df.isnull().sum().sum()} total')


Dataset: 33,621 filas × 20 columnas
Features: 19 (19 numéricas, 0 categóricas)
Target: 9,833 abandono (29.2%) · 23,788 éxito (70.8%)
Ratio desbalanceo: 1:2.4  (desbalanceo moderado)
Nulos: 12740 total


In [4]:
# ============================================================================
# CELDA 3: RANKING CONSOLIDADO DE FEATURES
# Combina: correlación punto-biserial + mutual info + d Cohen + AutoML
# ============================================================================

def cohens_d(a, b):
    sp = np.sqrt((np.std(a, ddof=1)**2 + np.std(b, ddof=1)**2) / 2)
    return abs(np.mean(a) - np.mean(b)) / sp if sp > 0 else 0

y = df[TARGET].values

# ── Punto-biserial (numéricas) ────────────────────────────────────────────────
pb = {}
for f in FEAT_NUM:
    r, p = pointbiserialr(df[f].fillna(df[f].median()), y)
    pb[f] = abs(r)

# ── Mutual information (todas) ────────────────────────────────────────────────
X_mi = df[FEATURES].copy()
for f in FEAT_CAT:
    X_mi[f] = pd.Categorical(X_mi[f]).codes
X_mi = X_mi.fillna(X_mi.median())
mi_vals = mutual_info_classif(X_mi, y, random_state=42, n_neighbors=5)
mi = dict(zip(FEATURES, mi_vals))

# ── d de Cohen (numéricas) ────────────────────────────────────────────────────
df_abnd = df[df[TARGET] == 1]
df_exit = df[df[TARGET] == 0]
cd = {}
for f in FEAT_NUM:
    cd[f] = cohens_d(df_abnd[f].dropna().values, df_exit[f].dropna().values)

# ── Ranking AutoML (desde parquet M06 si existe) ─────────────────────────────
automl_rank = {}
if RUTA_RANKING.exists():
    df_rk = pd.read_parquet(RUTA_RANKING)
    df_rk.columns = [c.lower().strip() for c in df_rk.columns]
    col_imp = next((c for c in ['importancia','importance','puntuacion_final','rango_medio']
                    if c in df_rk.columns), None)
    if col_imp and 'feature' in df_rk.columns:
        automl_rank = df_rk.set_index('feature')[col_imp].to_dict()

# ── Tabla consolidada ─────────────────────────────────────────────────────────
rows = []
for f in FEATURES:
    pb_v  = pb.get(f, 0)
    mi_v  = mi.get(f, 0)
    cd_v  = cd.get(f, 0)
    au_v  = automl_rank.get(f, 0)
    rows.append({'feature': f, 'punto_biserial': pb_v,
                 'mutual_info': mi_v, 'cohens_d': cd_v, 'automl_imp': au_v})

df_rank = pd.DataFrame(rows)

# Normalizar cada métrica 0-1 y calcular puntuación media
for col in ['punto_biserial','mutual_info','cohens_d','automl_imp']:
    mx = df_rank[col].max()
    df_rank[col+'_n'] = df_rank[col] / mx if mx > 0 else 0

cols_n = [c for c in df_rank.columns if c.endswith('_n')]
df_rank['score_final'] = df_rank[cols_n].mean(axis=1)
df_rank = df_rank.sort_values('score_final', ascending=False).reset_index(drop=True)
df_rank['rank'] = range(1, len(df_rank)+1)

print('Ranking consolidado (top 10):')
print(df_rank[['rank','feature','punto_biserial','mutual_info',
               'cohens_d','automl_imp','score_final']].head(10).to_string(index=False))

# Guardar para Fase 5
ruta_rank_out = ROOT / 'data' / 'automl' / 'ranking_final_fase4.parquet'
df_rank.to_parquet(ruta_rank_out, index=False)
print(f'\n✅ Ranking guardado: {ruta_rank_out}')


Ranking consolidado (top 10):
 rank                 feature  punto_biserial  mutual_info  cohens_d  automl_imp  score_final
    1            n_anios_beca        0.358962     0.079041  0.917689           0     0.663486
    2           nota_1er_anio        0.299822     0.110379  0.784787           0     0.650911
    3 cred_superados_anio_1er        0.273031     0.120868  0.642884           0     0.615290
    4             nota_acceso        0.287111     0.044059  0.717172           0     0.486464
    5               tuvo_beca        0.239965     0.030402  0.520559           0     0.371820
    6       situacion_laboral        0.209479     0.031938  0.474785           0     0.341294
    7       nota_selectividad        0.214400     0.023782  0.523510           0     0.341126
    8          anios_sin_beca        0.122912     0.034487  0.291715           0     0.236405
    9                    sexo        0.150021     0.015404  0.332746           0     0.226992
   10            edad_entrada 

In [5]:
# ============================================================================
# CELDA 4: DECISIONES DE PREPROCESAMIENTO — JUSTIFICADAS POR LOS DATOS
# ============================================================================

# ── Rangos reales de las features numéricas ───────────────────────────────────
rangos = []
for f in FEAT_NUM:
    vals = df[f].dropna()
    rangos.append({
        'feature': f,
        'min': vals.min(), 'max': vals.max(),
        'rango': vals.max() - vals.min(),
        'std': vals.std(),
        'skew': vals.skew(),
        'p99_p01': vals.quantile(0.99) - vals.quantile(0.01)
    })
df_rangos = pd.DataFrame(rangos).sort_values('rango', ascending=False)

# Feature con mayor rango (candidata a RobustScaler)
feat_max_rango = df_rangos.iloc[0]['feature']
feat_max_skew  = df_rangos.loc[df_rangos['skew'].abs().idxmax(), 'feature']

# ── Tabla de decisiones de escalado ──────────────────────────────────────────
# Regla: si rango > 100 o |skew| > 2 → RobustScaler preferible
# Regla: árboles (RF, XGB, LGB, CatBoost, EBM) → NO necesitan escalado
# Regla: lineales (LogReg, SVM, KNN, MLP) → StandardScaler obligatorio

escalado_decisiones = {
    'LogisticRegression':     'StandardScaler — sensible a escala, L1/L2 penalizan sin normalizar',
    'KNN':                    'StandardScaler — distancia euclidiana afectada directamente por escala',
    'LinearSVC':              'StandardScaler — margen de separación depende de la escala',
    'SGDClassifier':          'StandardScaler — gradiente descendente diverge sin normalización',
    'MLPClassifier':          'StandardScaler — activaciones saturan con valores grandes',
    'RandomForest':           'NO necesario — árbol de decisión invariante a la escala',
    'ExtraTrees':             'NO necesario — árbol de decisión invariante a la escala',
    'DecisionTree':           'NO necesario — árbol de decisión invariante a la escala',
    'XGBoost':                'NO necesario — gradient boosting invariante a la escala',
    'LightGBM':               'NO necesario — gradient boosting invariante a la escala',
    'CatBoost':               'NO necesario — maneja categóricas y escala nativamente',
    'HistGradientBoosting':   'NO necesario — gradient boosting invariante a la escala',
    'ExplainableBoostingMachine': 'NO necesario — basado en árboles, invariante a la escala',
}

print('Decisiones de escalado por algoritmo:')
for alg, dec in escalado_decisiones.items():
    print(f'  {alg:35s}: {dec}')

print(f'\n⚠️  Feature con mayor rango: {feat_max_rango}')
print(f'   → Candidata a RobustScaler si hay modelos sensibles a outliers')
print(f'⚠️  Feature con mayor asimetría: {feat_max_skew}')
print(f'   → Considerar transformación log1p antes de modelos lineales')

# ── Encoding categóricas ──────────────────────────────────────────────────────
enc_decisiones = {}
for f in FEAT_CAT:
    n_cats = df[f].nunique()
    if n_cats == 2:
        enc_decisiones[f] = f'Binary (0/1) — {n_cats} categorías'
    elif n_cats <= 8:
        enc_decisiones[f] = f'OrdinalEncoder o OneHot — {n_cats} categorías (bajo cardinalidad)'
    else:
        enc_decisiones[f] = f'OrdinalEncoder para árboles / OneHot para lineales — {n_cats} categorías'

print('\nDecisiones de encoding:')
for f, dec in enc_decisiones.items():
    print(f'  {f:30s}: {dec}')


Decisiones de escalado por algoritmo:
  LogisticRegression                 : StandardScaler — sensible a escala, L1/L2 penalizan sin normalizar
  KNN                                : StandardScaler — distancia euclidiana afectada directamente por escala
  LinearSVC                          : StandardScaler — margen de separación depende de la escala
  SGDClassifier                      : StandardScaler — gradiente descendente diverge sin normalización
  MLPClassifier                      : StandardScaler — activaciones saturan con valores grandes
  RandomForest                       : NO necesario — árbol de decisión invariante a la escala
  ExtraTrees                         : NO necesario — árbol de decisión invariante a la escala
  DecisionTree                       : NO necesario — árbol de decisión invariante a la escala
  XGBoost                            : NO necesario — gradient boosting invariante a la escala
  LightGBM                           : NO necesario — gradient boos

In [6]:
# ============================================================================
# CELDA 5: ESTRATEGIA DE BALANCEO DE CLASES
# ============================================================================

ratio = n_exit / n_abnd
print(f'Ratio mayoritaria/minoritaria: 1:{ratio:.2f}')
print()

if ratio < 2:
    nivel = 'leve'
    recomendacion_principal = 'class_weight="balanced"'
elif ratio < 4:
    nivel = 'moderado'
    recomendacion_principal = 'class_weight="balanced" + probar SMOTE'
else:
    nivel = 'alto'
    recomendacion_principal = 'SMOTE obligatorio'

print(f'Nivel de desbalanceo: {nivel}')
print(f'Recomendación principal: {recomendacion_principal}')
print()

# Las 3 estrategias a probar en Fase 5
estrategias = [
    {
        'nombre': 'Sin ajuste (baseline)',
        'descripcion': 'Entrena con el desbalanceo natural del dataset',
        'ventaja': 'Refleja la distribución real del problema',
        'riesgo': 'El modelo puede ignorar la clase minoritaria (abandono)',
        'cuando': 'Solo como baseline de comparación',
        'algoritmos': 'Todos'
    },
    {
        'nombre': 'class_weight="balanced"',
        'descripcion': 'Penaliza más los errores en la clase minoritaria',
        'ventaja': 'Sin modificar los datos, reproducible, rápido',
        'riesgo': 'Puede sobreestimar el abandono en casos límite',
        'cuando': 'Primera opción recomendada — desbalanceo moderado',
        'algoritmos': 'LogReg, RF, SVM, SGD, MLP (no XGB/LGB/CatBoost nativamente)'
    },
    {
        'nombre': 'SMOTE (Synthetic Minority Oversampling)',
        'descripcion': 'Genera ejemplos sintéticos de abandono en el espacio de features',
        'ventaja': 'Mejora recall en clase minoritaria sin perder información',
        'riesgo': 'SOLO aplicar sobre TRAIN — nunca sobre test (data leakage)',
        'cuando': 'Comparar con class_weight — usar el que dé mejor F1 en CV',
        'algoritmos': 'Todos (aplicado antes del fit, no dentro del modelo)'
    }
]

print('Las 3 estrategias a evaluar en Fase 5:')
for i, e in enumerate(estrategias, 1):
    print(f'  {i}. {e["nombre"]}')
    print(f'     Ventaja: {e["ventaja"]}')
    print(f'     Riesgo:  {e["riesgo"]}')
    print(f'     ⚠️  CRÍTICO: {e["cuando"]}')
    print()

print('⚠️  REGLA DE ORO: SMOTE y StandardScaler se ajustan SOLO sobre X_train.')
print('   Aplicarlos sobre X_test = data leakage. Todo dentro de Pipeline sklearn.')


Ratio mayoritaria/minoritaria: 1:2.42

Nivel de desbalanceo: moderado
Recomendación principal: class_weight="balanced" + probar SMOTE

Las 3 estrategias a evaluar en Fase 5:
  1. Sin ajuste (baseline)
     Ventaja: Refleja la distribución real del problema
     Riesgo:  El modelo puede ignorar la clase minoritaria (abandono)
     ⚠️  CRÍTICO: Solo como baseline de comparación

  2. class_weight="balanced"
     Ventaja: Sin modificar los datos, reproducible, rápido
     Riesgo:  Puede sobreestimar el abandono en casos límite
     ⚠️  CRÍTICO: Primera opción recomendada — desbalanceo moderado

  3. SMOTE (Synthetic Minority Oversampling)
     Ventaja: Mejora recall en clase minoritaria sin perder información
     Riesgo:  SOLO aplicar sobre TRAIN — nunca sobre test (data leakage)
     ⚠️  CRÍTICO: Comparar con class_weight — usar el que dé mejor F1 en CV

⚠️  REGLA DE ORO: SMOTE y StandardScaler se ajustan SOLO sobre X_train.
   Aplicarlos sobre X_test = data leakage. Todo dentro de Pipe

In [7]:
# ============================================================================
# CELDA 6: MULTICOLINEALIDAD — PARES PROBLEMÁTICOS PARA MODELOS LINEALES
# ============================================================================
import warnings
warnings.filterwarnings('ignore')

# Correlación de Pearson entre features numéricas
df_num = df[FEAT_NUM].fillna(df[FEAT_NUM].median())
corr_matrix = df_num.corr().abs()

# Pares con |r| >= 0.7 (umbral de multicolinealidad)
umbral = 0.7
pares_altos = []
for i, f1 in enumerate(FEAT_NUM):
    for j, f2 in enumerate(FEAT_NUM):
        if j <= i: continue
        r = corr_matrix.loc[f1, f2]
        if r >= umbral:
            pares_altos.append({'f1': f1, 'f2': f2, 'r': r})

df_pares = pd.DataFrame(pares_altos).sort_values('r', ascending=False)

if len(df_pares) > 0:
    print(f'Pares con |r| >= {umbral} (riesgo de multicolinealidad):')
    for _, row in df_pares.iterrows():
        print(f'  {row["f1"]:30s} × {row["f2"]:30s}  r={row["r"]:.3f}')
    print()
    print('Implicación para Fase 5:')
    print('  - Modelos lineales (LogReg, SVM, MLP): considerar eliminar una de cada par')
    print('  - Modelos de árboles (RF, XGB, LGB, CatBoost): no afecta, pueden manejarlo')
    print('  - Solución recomendada: mantener todas y dejar que Lasso/L1 penalice automáticamente')
else:
    print(f'✅ No hay pares con |r| >= {umbral}. Sin problemas de multicolinealidad severa.')

# Correlaciones moderadas (0.5 <= r < 0.7) — solo informativo
pares_mod = []
for i, f1 in enumerate(FEAT_NUM):
    for j, f2 in enumerate(FEAT_NUM):
        if j <= i: continue
        r = corr_matrix.loc[f1, f2]
        if 0.5 <= r < umbral:
            pares_mod.append({'f1': f1, 'f2': f2, 'r': r})

if pares_mod:
    print(f'\nPares con correlación moderada (0.5–{umbral}) — solo informativo:')
    for p in sorted(pares_mod, key=lambda x: -x['r'])[:5]:
        print(f'  {p["f1"]:30s} × {p["f2"]:30s}  r={p["r"]:.3f}')


Pares con |r| >= 0.7 (riesgo de multicolinealidad):
  indicador_interrupcion         × anios_gap                       r=0.833
  nota_acceso                    × nota_selectividad               r=0.725

Implicación para Fase 5:
  - Modelos lineales (LogReg, SVM, MLP): considerar eliminar una de cada par
  - Modelos de árboles (RF, XGB, LGB, CatBoost): no afecta, pueden manejarlo
  - Solución recomendada: mantener todas y dejar que Lasso/L1 penalice automáticamente

Pares con correlación moderada (0.5–0.7) — solo informativo:
  tuvo_beca                      × n_anios_beca                    r=0.681
  nota_1er_anio                  × nota_selectividad               r=0.513


In [8]:
# ============================================================================
# CELDA 7: HALLAZGO PRINCIPAL — PATRÓN ASIMÉTRICO DEL ABANDONO
# ============================================================================

# Calcular d de Cohen para todas las features y verificar el patrón
patron_items = []
for f in FEAT_NUM:
    a = df_abnd[f].dropna().values
    e = df_exit[f].dropna().values
    if len(a) < 10 or len(e) < 10: continue
    d   = cohens_d(a, e)
    dir_mayor = 'éxito' if e.mean() > a.mean() else 'abandono'
    patron_items.append({'feature': f, 'd': d, 'dir_mayor': dir_mayor})

df_patron = pd.DataFrame(patron_items).sort_values('d', ascending=False)

n_exito_mayor   = sum(1 for _, r in df_patron.iterrows() 
                      if r['d'] >= 0.5 and r['dir_mayor'] == 'éxito')
n_abnd_mayor    = sum(1 for _, r in df_patron.iterrows() 
                      if r['d'] >= 0.5 and r['dir_mayor'] == 'abandono')
n_sin_efecto    = sum(1 for _, r in df_patron.iterrows() if r['d'] < 0.5)

print('=' * 60)
print('HALLAZGO PRINCIPAL: PATRÓN ASIMÉTRICO DEL ABANDONO')
print('=' * 60)
print()
print(f'Features con efecto medio/grande (d >= 0.5):')
print(f'  🟢 Éxito mayor (protectoras):  {n_exito_mayor}')
print(f'  🔴 Abandono mayor (de riesgo): {n_abnd_mayor}')
print(f'  🟡 Efecto débil (d < 0.5):     {n_sin_efecto}')
print()
print('INTERPRETACIÓN:')
print(f'  El abandono en la UJI NO se explica por la presencia de')
print(f'  factores negativos propios, sino por el DÉFICIT de factores')
print(f'  protectores: menos créditos superados, peor nota en primer')
print(f'  año, menos años con beca.')
print()
print('IMPLICACIÓN PARA FASE 5 Y 6:')
print(f'  - Los valores SHAP negativos (reducen probabilidad de abandono)')
print(f'    serán los más importantes')
print(f'  - La intervención preventiva debe focalizarse en refuerzo')
print(f'    temprano en primer curso (detección en semestre 1)')
print(f'  - Referencia: Tinto (1987) — integración académica y social')
print()
print('Features protectoras clave (d >= 0.5, éxito mayor):')
for _, r in df_patron[
    (df_patron['d'] >= 0.5) & (df_patron['dir_mayor'] == 'éxito')
].iterrows():
    mag = 'Grande' if r['d'] >= 0.8 else 'Medio'
    print(f'  🟢 {r["feature"]:35s} d={r["d"]:.2f} ({mag})')


HALLAZGO PRINCIPAL: PATRÓN ASIMÉTRICO DEL ABANDONO

Features con efecto medio/grande (d >= 0.5):
  🟢 Éxito mayor (protectoras):  6
  🔴 Abandono mayor (de riesgo): 0
  🟡 Efecto débil (d < 0.5):     13

INTERPRETACIÓN:
  El abandono en la UJI NO se explica por la presencia de
  factores negativos propios, sino por el DÉFICIT de factores
  protectores: menos créditos superados, peor nota en primer
  año, menos años con beca.

IMPLICACIÓN PARA FASE 5 Y 6:
  - Los valores SHAP negativos (reducen probabilidad de abandono)
    serán los más importantes
  - La intervención preventiva debe focalizarse en refuerzo
    temprano en primer curso (detección en semestre 1)
  - Referencia: Tinto (1987) — integración académica y social

Features protectoras clave (d >= 0.5, éxito mayor):
  🟢 n_anios_beca                        d=0.92 (Grande)
  🟢 nota_1er_anio                       d=0.78 (Medio)
  🟢 nota_acceso                         d=0.72 (Medio)
  🟢 cred_superados_anio_1er             d=0.64 (Medi

In [9]:
# ============================================================================
# CELDA 8: CHECKLIST DE ENTRADA A FASE 5
# ============================================================================

checklist = [
    # (descripción, check_fn, crítico)
    ('Dataset D_strict cargado correctamente',
     lambda: df.shape[0] > 30000 and df.shape[1] >= 19, True),
    ('Target binario (0/1) sin nulos',
     lambda: df[TARGET].isnull().sum() == 0 and df[TARGET].nunique() == 2, True),
    ('Sin columnas con leakage temporal',
     lambda: not any(c in df.columns for c in 
                     ['anio_fin','fecha_fin','ultimo_curso','estabilidad_academica',
                      'mejora_notas','progresion_rendimiento']), True),
    ('Sin columnas constantes',
     lambda: all(df[c].nunique() > 1 for c in df.columns), True),
    ('Nulos por debajo del 15% en todas las features',
     lambda: all(df[c].isnull().mean() < 0.15 for c in FEATURES), True),
    ('Ranking final de features generado',
     lambda: (ROOT / 'data' / 'automl' / 'ranking_final_fase4.parquet').exists(), True),
    ('HTML Fase 4 generados (M01-M08)',
     lambda: len(list((ROOT / 'docs' / 'html' / 'fase4').glob('*.html'))) >= 8, False),
    ('Train/test split: PENDIENTE',
     lambda: False, False),
    ('Pipeline preprocesamiento: PENDIENTE',
     lambda: False, False),
    ('Estrategia balanceo decidida: class_weight + SMOTE para comparar',
     lambda: True, False),
]

print('=' * 65)
print('CHECKLIST ENTRADA A FASE 5')
print('=' * 65)
n_ok = 0; n_warn = 0; n_fail = 0
for desc, fn, critico in checklist:
    try:
        ok = fn()
    except Exception:
        ok = False
    if ok:
        print(f'  ✅ {desc}')
        n_ok += 1
    elif critico:
        print(f'  ❌ {desc}  ← CRÍTICO')
        n_fail += 1
    else:
        print(f'  ⏳ {desc}  ← PENDIENTE')
        n_warn += 1

print()
print(f'Resultado: {n_ok} OK · {n_warn} pendientes · {n_fail} críticos')
if n_fail == 0:
    print('✅ FASE 4 COMPLETADA — dataset listo para entrar a Fase 5')
else:
    print('❌ Resolver los puntos críticos antes de continuar con Fase 5')


CHECKLIST ENTRADA A FASE 5
  ✅ Dataset D_strict cargado correctamente
  ✅ Target binario (0/1) sin nulos
  ✅ Sin columnas con leakage temporal
  ✅ Sin columnas constantes
  ✅ Nulos por debajo del 15% en todas las features
  ✅ Ranking final de features generado
  ✅ HTML Fase 4 generados (M01-M08)
  ⏳ Train/test split: PENDIENTE  ← PENDIENTE
  ⏳ Pipeline preprocesamiento: PENDIENTE  ← PENDIENTE
  ✅ Estrategia balanceo decidida: class_weight + SMOTE para comparar

Resultado: 8 OK · 2 pendientes · 0 críticos
✅ FASE 4 COMPLETADA — dataset listo para entrar a Fase 5


In [10]:
# ============================================================================
# CELDA 9: GENERACIÓN HTML
# ============================================================================
import plotly.graph_objects as go
import plotly.express as px

# ── Gráfico 1: Ranking consolidado ────────────────────────────────────────────
fig_rank = go.Figure()
top_n = min(15, len(df_rank))
df_top = df_rank.head(top_n)

for col, nombre, color in [
    ('punto_biserial_n', 'Punto-biserial', '#3182ce'),
    ('mutual_info_n',    'Mutual Info',    '#38a169'),
    ('cohens_d_n',       'd de Cohen',     '#e53e3e'),
    ('automl_imp_n',     'AutoML',         '#d69e2e'),
]:
    if col in df_top.columns and df_top[col].sum() > 0:
        fig_rank.add_trace(go.Bar(
            name=nombre,
            x=df_top['feature'].str.replace('_',' '),
            y=df_top[col],
            marker_color=color,
            opacity=0.8
        ))

fig_rank.update_layout(
    barmode='group',
    title=dict(text='Ranking consolidado de features — 4 métricas normalizadas (0–1)', 
               font=dict(size=13), x=0.5),
    height=420,
    xaxis=dict(tickangle=-35, tickfont=dict(size=10)),
    yaxis=dict(title='Importancia normalizada (0–1)', showgrid=True, gridcolor='#e2e8f0'),
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='center', x=0.5),
    paper_bgcolor='white', plot_bgcolor='white',
    margin=dict(t=70, b=120, l=50, r=20)
)
IMG_RANK = fig_rank.to_html(full_html=False, include_plotlyjs='cdn')

# ── Gráfico 2: Score final — barras horizontales ──────────────────────────────
fig_score = go.Figure(go.Bar(
    x=df_rank.head(top_n)['score_final'],
    y=df_rank.head(top_n)['feature'].str.replace('_',' '),
    orientation='h',
    marker=dict(
        color=df_rank.head(top_n)['score_final'],
        colorscale=[[0,'#ebf8ff'],[0.5,'#3182ce'],[1,'#1a365d']],
        showscale=False
    ),
    text=df_rank.head(top_n)['score_final'].round(3),
    textposition='outside',
    hovertemplate='<b>%{y}</b><br>Score final: %{x:.3f}<extra></extra>'
))
fig_score.update_layout(
    title=dict(text='Score final consolidado — media de las 4 métricas normalizadas',
               font=dict(size=13), x=0.5),
    height=max(350, top_n * 30),
    xaxis=dict(title='Score (0–1)', showgrid=True, gridcolor='#e2e8f0', range=[0, 1.1]),
    yaxis=dict(autorange='reversed', tickfont=dict(size=11)),
    paper_bgcolor='white', plot_bgcolor='white',
    margin=dict(t=60, b=50, l=175, r=70)
)
IMG_SCORE = fig_score.to_html(full_html=False, include_plotlyjs='cdn')

# ── Gráfico 3: d de Cohen — semáforo visual ───────────────────────────────────
df_cohen = df_rank[df_rank['cohens_d'] > 0].sort_values('cohens_d', ascending=False).head(top_n)
colors_cohen = ['#e53e3e' if d >= 0.8 else '#d69e2e' if d >= 0.5 else '#a0aec0'
                for d in df_cohen['cohens_d']]
fig_cohen = go.Figure(go.Bar(
    x=df_cohen['cohens_d'],
    y=df_cohen['feature'].str.replace('_',' '),
    orientation='h',
    marker_color=colors_cohen,
    text=df_cohen['cohens_d'].round(2),
    textposition='outside',
    hovertemplate='<b>%{y}</b><br>d Cohen: %{x:.2f}<extra></extra>'
))
fig_cohen.add_vline(x=0.2, line=dict(color='#a0aec0', dash='dash', width=1))
fig_cohen.add_vline(x=0.5, line=dict(color='#d69e2e', dash='dash', width=1.5))
fig_cohen.add_vline(x=0.8, line=dict(color='#e53e3e', dash='dash', width=1.5))
fig_cohen.add_annotation(x=0.2, y=-0.8, text='0.2<br>pequeño', showarrow=False,
                          font=dict(size=9, color='#718096'))
fig_cohen.add_annotation(x=0.5, y=-0.8, text='0.5<br>medio', showarrow=False,
                          font=dict(size=9, color='#744210'))
fig_cohen.add_annotation(x=0.8, y=-0.8, text='0.8<br>grande', showarrow=False,
                          font=dict(size=9, color='#9b2c2c'))
fig_cohen.update_layout(
    title=dict(text='d de Cohen por feature — magnitud del efecto entre grupos',
               font=dict(size=13), x=0.5),
    height=max(350, len(df_cohen) * 30),
    xaxis=dict(title='d de Cohen', showgrid=True, gridcolor='#e2e8f0'),
    yaxis=dict(autorange='reversed', tickfont=dict(size=11)),
    paper_bgcolor='white', plot_bgcolor='white',
    margin=dict(t=60, b=70, l=175, r=70)
)
IMG_COHEN = fig_cohen.to_html(full_html=False, include_plotlyjs='cdn')

# ── Tabla ranking HTML ────────────────────────────────────────────────────────
def score_bar(v, max_v=1.0):
    pct = min(int(v / max_v * 100), 100)
    return (f'<div style="display:flex;align-items:center;gap:6px;">'
            f'<div style="width:{pct}px;height:10px;background:#3182ce;'
            f'border-radius:3px;min-width:2px;"></div>'
            f'<span style="font-size:11px;">{v:.3f}</span></div>')

filas_rank = ''
for _, row in df_rank.head(15).iterrows():
    d = row['cohens_d']
    if d >= 0.8:   ico = '🔴 Grande'
    elif d >= 0.5: ico = '🟡 Medio'
    elif d >= 0.2: ico = '⚪ Pequeño'
    else:          ico = '⬜ Negligible'
    filas_rank += (
        f'<tr>'
        f'<td style="padding:7px 10px;text-align:center;font-weight:700;">{int(row["rank"])}</td>'
        f'<td style="padding:7px 10px;font-weight:600;">{row["feature"].replace("_"," ")}</td>'
        f'<td style="padding:7px 10px;">{score_bar(row["punto_biserial_n"])}</td>'
        f'<td style="padding:7px 10px;">{score_bar(row["mutual_info_n"])}</td>'
        f'<td style="padding:7px 10px;">{score_bar(row["cohens_d_n"])}</td>'
        f'<td style="padding:7px 10px;">{score_bar(row["automl_imp_n"])}</td>'
        f'<td style="padding:7px 10px;">{score_bar(row["score_final"])}</td>'
        f'<td style="padding:7px 10px;font-size:12px;">{ico}</td>'
        f'</tr>'
    )

tabla_rank_html = (
    '<div style="overflow-x:auto;">'
    '<table style="width:100%;border-collapse:collapse;">'
    '<thead><tr style="background:#2d3748;color:#fff;">'
    '<th style="padding:9px 10px;">#</th>'
    '<th style="padding:9px 10px;text-align:left;">Feature</th>'
    '<th style="padding:9px 10px;">Punto-biserial</th>'
    '<th style="padding:9px 10px;">Mutual Info</th>'
    '<th style="padding:9px 10px;">d Cohen</th>'
    '<th style="padding:9px 10px;">AutoML</th>'
    '<th style="padding:9px 10px;">Score final</th>'
    '<th style="padding:9px 10px;">Magnitud</th>'
    '</tr></thead>'
    f'<tbody>{filas_rank}</tbody>'
    '</table></div>'
    '<p style="color:#718096;font-size:11px;margin-top:6px;">'
    'Score final = media normalizada de las 4 métricas. '
    'Magnitud según d de Cohen (1988).</p>'
)

# ── Tabla decisiones preprocesamiento ────────────────────────────────────────
filas_prep = ''
for alg, dec in escalado_decisiones.items():
    necesita = 'NO' in dec
    bg = 'background:#f0fff4;' if necesita else 'background:#ebf8ff;'
    filas_prep += (
        f'<tr style="{bg}">'
        f'<td style="padding:7px 12px;font-weight:600;">{alg}</td>'
        f'<td style="padding:7px 12px;font-size:13px;">{dec}</td>'
        f'</tr>'
    )

tabla_prep_html = (
    '<table style="width:100%;border-collapse:collapse;">'
    '<thead><tr style="background:#2d3748;color:#fff;">'
    '<th style="padding:9px 12px;text-align:left;">Algoritmo</th>'
    '<th style="padding:9px 12px;text-align:left;">Escalado necesario</th>'
    '</tr></thead>'
    f'<tbody>{filas_prep}</tbody>'
    '</table>'
)

# ── Tabla balanceo ────────────────────────────────────────────────────────────
filas_bal = ''
colores_bal = ['#fff5f5', '#ebf8ff', '#f0fff4']
for i, e in enumerate(estrategias):
    bg = colores_bal[i]
    filas_bal += (
        f'<tr style="background:{bg};">'
        f'<td style="padding:8px 12px;font-weight:700;">{e["nombre"]}</td>'
        f'<td style="padding:8px 12px;font-size:13px;">{e["descripcion"]}</td>'
        f'<td style="padding:8px 12px;font-size:13px;color:#276749;">{e["ventaja"]}</td>'
        f'<td style="padding:8px 12px;font-size:13px;color:#9b2c2c;">{e["riesgo"]}</td>'
        f'<td style="padding:8px 12px;font-size:13px;font-style:italic;">{e["cuando"]}</td>'
        f'</tr>'
    )

tabla_bal_html = (
    '<div style="overflow-x:auto;">'
    '<table style="width:100%;border-collapse:collapse;">'
    '<thead><tr style="background:#2d3748;color:#fff;">'
    '<th style="padding:9px 12px;">Estrategia</th>'
    '<th style="padding:9px 12px;">Descripción</th>'
    '<th style="padding:9px 12px;">Ventaja</th>'
    '<th style="padding:9px 12px;">Riesgo</th>'
    '<th style="padding:9px 12px;">Cuándo usar</th>'
    '</tr></thead>'
    f'<tbody>{filas_bal}</tbody>'
    '</table></div>'
    f'<div style="background:#fff5f5;border:1px solid #feb2b2;border-radius:6px;'
    f'padding:12px 16px;margin-top:10px;font-size:13px;color:#2d3748;">'
    f'⚠️ <b>REGLA DE ORO:</b> SMOTE y StandardScaler se ajustan <b>SOLO sobre X_train</b>. '
    f'Aplicarlos sobre X_test = data leakage. Todo dentro de <code>Pipeline</code> sklearn.'
    f'</div>'
)

# ── Tabla multicolinealidad ───────────────────────────────────────────────────
if len(df_pares) > 0:
    filas_mc = ''
    for _, row in df_pares.iterrows():
        nivel_mc = '🔴 Alta' if row['r'] >= 0.85 else '🟡 Moderada-alta'
        filas_mc += (
            f'<tr>'
            f'<td style="padding:7px 12px;">{row["f1"].replace("_"," ")}</td>'
            f'<td style="padding:7px 12px;">{row["f2"].replace("_"," ")}</td>'
            f'<td style="padding:7px 12px;text-align:center;font-weight:700;">{row["r"]:.3f}</td>'
            f'<td style="padding:7px 12px;">{nivel_mc}</td>'
            f'</tr>'
        )
    tabla_mc_html = (
        '<table style="width:100%;border-collapse:collapse;">'
        '<thead><tr style="background:#2d3748;color:#fff;">'
        '<th style="padding:9px 12px;text-align:left;">Feature 1</th>'
        '<th style="padding:9px 12px;text-align:left;">Feature 2</th>'
        '<th style="padding:9px 12px;">|r|</th>'
        '<th style="padding:9px 12px;">Nivel</th>'
        '</tr></thead>'
        f'<tbody>{filas_mc}</tbody>'
        '</table>'
    )
else:
    tabla_mc_html = (
        '<div style="background:#f0fff4;border:1px solid #9ae6b4;border-radius:6px;'
        'padding:14px 16px;font-size:13px;color:#276749;">'
        '✅ No se detectan pares con |r| ≥ 0.7. Sin problemas de multicolinealidad severa.'
        '</div>'
    )

# ── Checklist HTML ────────────────────────────────────────────────────────────
filas_check = ''
for desc, fn, critico in checklist:
    try: ok = fn()
    except: ok = False
    if ok:
        ico, bg = '✅', 'background:#f0fff4;'
    elif critico:
        ico, bg = '❌', 'background:#fff5f5;'
    else:
        ico, bg = '⏳', 'background:#fffbeb;'
    filas_check += (
        f'<tr style="{bg}">'
        f'<td style="padding:8px 12px;font-size:16px;">{ico}</td>'
        f'<td style="padding:8px 12px;font-size:13px;">{desc}</td>'
        f'</tr>'
    )

tabla_check_html = (
    '<table style="width:100%;border-collapse:collapse;">'
    f'<tbody>{filas_check}</tbody>'
    '</table>'
)

# ── Resumen algoritmos Fase 5 ─────────────────────────────────────────────────
algoritmos_f5 = [
    ('Familia 1 — Lineales / simples', COLOR_EXIT,
     'LogisticRegression (L1+L2) · KNN · Naive Bayes · SGDClassifier · LinearSVC',
     'Suelo mínimo. LogReg es el árbitro: si Gradient Boosting no lo supera, hay un problema en los datos.'),
    ('Familia 2 — Árboles clásicos', '#38a169',
     'DecisionTree · RandomForest · ExtraTreesClassifier',
     'RandomForest salió bien en AutoML. DecisionTree da reglas directamente interpretables para el tribunal.'),
    ('Familia 3 — Gradient Boosting', COLOR_ABND,
     'XGBoost · LightGBM · CatBoost · HistGradientBoosting',
     f'Los ganadores AutoML (CatBoost F1=0.7970, AUC≈0.93). Aquí está la batalla real de Fase 5.'),
    ('Familia 4 — Red neuronal ligera', '#9f7aea',
     'MLPClassifier (1–3 capas)',
     'Recomendado por director. Sin Keras/TensorFlow — MLPClassifier suficiente para 33k filas.'),
    ('Familia 5 — Interpretable por diseño', '#d69e2e',
     'ExplainableBoostingMachine (Microsoft InterpretML)',
     'Modelo nº16. Potente Y totalmente interpretable. Conexión directa con Fase 6 SHAP.'),
    ('SVM (condicionado)', '#718096',
     'LinearSVC + SVM-RBF sobre muestra 20% con timeout 30min',
     'Con 33k filas RBF puede tardar horas. LinearSVC es la alternativa rápida. '
     'Si SVM-RBF no termina en 30min, queda comentado en el notebook con justificación.'),
    ('Ensemble final', '#2d3748',
     'VotingClassifier (soft, top 3) · StackingClassifier (meta-learner LogReg)',
     'Stacking casi siempre mejora 1–2 puntos F1. Si no mejora, también es resultado válido.'),
]

filas_alg = ''
for familia, color, modelos, nota in algoritmos_f5:
    filas_alg += (
        f'<tr>'
        f'<td style="padding:9px 12px;border-left:4px solid {color};'
        f'font-weight:700;font-size:13px;">{familia}</td>'
        f'<td style="padding:9px 12px;font-size:13px;">{modelos}</td>'
        f'<td style="padding:9px 12px;font-size:12px;color:#4a5568;">{nota}</td>'
        f'</tr>'
    )

tabla_alg_html = (
    '<table style="width:100%;border-collapse:collapse;">'
    '<thead><tr style="background:#2d3748;color:#fff;">'
    '<th style="padding:9px 12px;text-align:left;">Familia</th>'
    '<th style="padding:9px 12px;text-align:left;">Modelos</th>'
    '<th style="padding:9px 12px;text-align:left;">Justificación</th>'
    '</tr></thead>'
    f'<tbody>{filas_alg}</tbody>'
    '</table>'
    f'<p style="color:#718096;font-size:11px;margin-top:6px;">'
    f'Total estimado: ~18 modelos base + 2 ensemble = 20 modelos · 5-fold CV = 100 entrenamientos</p>'
)

# ── KPIs ──────────────────────────────────────────────────────────────────────
kpis_html = (
    '<div style="display:grid;grid-template-columns:repeat(auto-fit,minmax(140px,1fr));'
    'gap:14px;margin-bottom:28px;">'
    f'<div style="background:#ebf8ff;border-left:4px solid {COLOR_EXIT};'
    'padding:14px 18px;border-radius:8px;">'
    '<div style="font-size:11px;color:#4a5568;text-transform:uppercase;'
    'letter-spacing:.6px;margin-bottom:4px;">Estudiantes</div>'
    f'<div style="font-size:26px;font-weight:700;">{formato_numero_es(n_total)}</div>'
    f'<div style="font-size:12px;color:#4a5568;">dataset D_strict</div></div>'
    f'<div style="background:#fff5f5;border-left:4px solid {COLOR_ABND};'
    'padding:14px 18px;border-radius:8px;">'
    '<div style="font-size:11px;color:#4a5568;text-transform:uppercase;'
    'letter-spacing:.6px;margin-bottom:4px;">Tasa abandono</div>'
    f'<div style="font-size:26px;font-weight:700;">{pct_abnd:.1f}%</div>'
    f'<div style="font-size:12px;color:#4a5568;">{formato_numero_es(n_abnd)} estudiantes</div></div>'
    '<div style="background:#f0fff4;border-left:4px solid #38a169;'
    'padding:14px 18px;border-radius:8px;">'
    '<div style="font-size:11px;color:#4a5568;text-transform:uppercase;'
    'letter-spacing:.6px;margin-bottom:4px;">Features</div>'
    f'<div style="font-size:26px;font-weight:700;">{len(FEATURES)}</div>'
    f'<div style="font-size:12px;color:#4a5568;">{len(FEAT_NUM)} num · {len(FEAT_CAT)} cat</div></div>'
    '<div style="background:#fffbeb;border-left:4px solid #d69e2e;'
    'padding:14px 18px;border-radius:8px;">'
    '<div style="font-size:11px;color:#4a5568;text-transform:uppercase;'
    'letter-spacing:.6px;margin-bottom:4px;">Desbalanceo</div>'
    f'<div style="font-size:26px;font-weight:700;">1:{ratio_bal:.1f}</div>'
    f'<div style="font-size:12px;color:#4a5568;">{nivel} — {recomendacion_principal}</div></div>'
    '</div>'
)

# ── Cuerpo principal ──────────────────────────────────────────────────────────
cuerpo = (
    kpis_html
    + '<h2 style="font-size:18px;font-weight:700;color:#1a202c;margin:32px 0 8px;">'
    '📊 Ranking consolidado de features (4 métricas)</h2>'
    + '<p style="color:#4a5568;font-size:13px;margin-bottom:12px;">'
    'Score final = media normalizada de punto-biserial, mutual info, d de Cohen y AutoML feature importance. '
    'Cuanto más consistente aparece una feature en las 4 métricas, más robusta es su importancia.</p>'
    + IMG_RANK
    + IMG_SCORE
    + tabla_rank_html
    + '<h2 style="font-size:18px;font-weight:700;color:#1a202c;margin:32px 0 8px;">'
    '🔬 Magnitud del efecto — d de Cohen</h2>'
    + '<p style="color:#4a5568;font-size:13px;margin-bottom:12px;">'
    'Mide cuántas desviaciones típicas separan los grupos. '
    'Rojo ≥ 0.8 (grande) · Naranja ≥ 0.5 (medio) · Gris &lt; 0.5 (pequeño o negligible).</p>'
    + IMG_COHEN
    + '<h2 style="font-size:18px;font-weight:700;color:#1a202c;margin:32px 0 8px;">'
    '💡 Hallazgo principal — patrón asimétrico del abandono</h2>'
    + f'<div style="background:#fffbeb;border:1px solid #f6e05e;border-radius:8px;'
    f'padding:18px 20px;margin-bottom:20px;">'
    f'<p style="font-size:14px;color:#2d3748;margin:0 0 10px;">'
    f'El abandono en la UJI <strong>no se explica por la presencia de factores negativos propios</strong>, '
    f'sino por el <strong>déficit de factores protectores</strong>: '
    f'menos créditos superados en primer año, peor nota, menos años con beca.</p>'
    f'<p style="font-size:14px;color:#2d3748;margin:0 0 10px;">'
    f'De las {len(FEAT_NUM)} features numéricas analizadas, '
    f'<strong>{n_exito_mayor} son protectoras</strong> con efecto medio/grande (d≥0.5) '
    f'y <strong>{n_abnd_mayor} son factores de riesgo propios</strong>. '
    f'Este patrón orienta la intervención hacia el <strong>refuerzo temprano en primer curso</strong>.</p>'
    f'<p style="font-size:13px;color:#718096;margin:0;">'
    f'Referencia: Tinto, V. (1987). <em>Leaving College</em>. University of Chicago Press.</p>'
    f'</div>'
    + '<h2 style="font-size:18px;font-weight:700;color:#1a202c;margin:32px 0 8px;">'
    '⚙️ Decisiones de preprocesamiento para Fase 5</h2>'
    + '<p style="color:#4a5568;font-size:13px;margin-bottom:12px;">'
    'El escalado depende del algoritmo, no del dataset. '
    'Las features tienen rangos muy distintos (nota 0–10 vs cred_superados 0–230), '
    'lo que hace el escalado crítico para los modelos lineales.</p>'
    + tabla_prep_html
    + '<h2 style="font-size:18px;font-weight:700;color:#1a202c;margin:32px 0 8px;">'
    '⚖️ Estrategia de balanceo de clases</h2>'
    + f'<p style="color:#4a5568;font-size:13px;margin-bottom:12px;">'
    f'Desbalanceo 1:{ratio_bal:.1f} ({nivel}). '
    f'Se evaluarán 3 estrategias en Fase 5 — la que dé mejor F1 en validación cruzada será la definitiva.</p>'
    + tabla_bal_html
    + '<h2 style="font-size:18px;font-weight:700;color:#1a202c;margin:32px 0 8px;">'
    '🔗 Multicolinealidad — pares problemáticos</h2>'
    + '<p style="color:#4a5568;font-size:13px;margin-bottom:12px;">'
    'Pares con |r| ≥ 0.7. Relevante para modelos lineales. '
    'Los modelos de árboles son robustos a multicolinealidad.</p>'
    + tabla_mc_html
    + '<h2 style="font-size:18px;font-weight:700;color:#1a202c;margin:32px 0 8px;">'
    '🤖 Plan de algoritmos — Fase 5</h2>'
    + '<p style="color:#4a5568;font-size:13px;margin-bottom:12px;">'
    'Clasificación supervisada binaria. ~18 modelos base + 2 ensemble. '
    'Referencia AutoML: CatBoost F1=0.7970, AUC≈0.93 — este es el suelo a superar.</p>'
    + tabla_alg_html
    + '<h2 style="font-size:18px;font-weight:700;color:#1a202c;margin:32px 0 8px;">'
    '✅ Checklist de entrada a Fase 5</h2>'
    + tabla_check_html
    + '<div style="background:#ebf8ff;border:1px solid #bee3f8;border-radius:8px;'
    'padding:18px 20px;margin-top:28px;">'
    '<h3 style="font-size:15px;font-weight:700;color:#2b6cb0;margin:0 0 12px;">'
    '🚀 Próximo paso: f5_m01_preparacion.ipynb</h3>'
    '<ul style="color:#2d3748;font-size:13px;margin:0;padding-left:18px;">'
    '<li>Train/test split estratificado 80/20 (semilla fija)</li>'
    '<li>Pipeline sklearn: imputer → scaler → encoder</li>'
    '<li>Versión A: sin balanceo · Versión B: class_weight · Versión C: SMOTE</li>'
    '<li>Exportar X_train, X_test, y_train, y_test como parquets</li>'
    '<li>Verificar distribución del target en train y test (~29.7% en ambos)</li>'
    '</ul>'
    '</div>'
)

nav_fases, nav_modulos = generar_html_navegacion_completa(
    fase_activa='fase4', modulo_activo='m09')
html_final = render_base_html(
    titulo=TITULO, icono='📋', subtitulo=SUBTITULO,
    nav_fases=nav_fases, nav_modulos=nav_modulos,
    contenido=cuerpo,
    notebook_nombre='f4_m09_conclusiones_eda.ipynb',
    notebook_carpeta='fase4_eda'
)
ruta_html = RUTA_FASE4_HTML / 'm09_conclusiones_eda.html'
guardar_html(html_final, ruta_html)

print('=' * 60)
print('✅ F4-M09 COMPLETADO')
print('=' * 60)
print(f'  Features analizadas : {len(FEATURES)}')
print(f'  Top feature         : {df_rank.iloc[0]["feature"]} (score={df_rank.iloc[0]["score_final"]:.3f})')
print(f'  Protectoras (d≥0.5) : {n_exito_mayor}')
print(f'  Riesgo (d≥0.5)      : {n_abnd_mayor}')
print(f'  Multicolinealidad   : {len(df_pares)} pares con |r|≥0.7')
print(f'  Ranking guardado    : ranking_final_fase4.parquet')
print(f'  HTML                : {ruta_html}')
print()
print('📌 Siguiente: f5_m01_preparacion.ipynb')


✅ HTML guardado: C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\docs\html\fase4\m09_conclusiones_eda.html
✅ F4-M09 COMPLETADO
  Features analizadas : 19
  Top feature         : n_anios_beca (score=0.663)
  Protectoras (d≥0.5) : 6
  Riesgo (d≥0.5)      : 0
  Multicolinealidad   : 2 pares con |r|≥0.7
  Ranking guardado    : ranking_final_fase4.parquet
  HTML                : C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\docs\html\fase4\m09_conclusiones_eda.html

📌 Siguiente: f5_m01_preparacion.ipynb
